# SOTA SFT Eval + Gated Export v2 (Notebook F_sota)

Plan: `fine_tuning/notebooks/PLAN_FABLE5_TO_IMPROVE_FN.md`

- Greedy eval on frozen `qa_test_frozen.jsonl`
- **EXPORT=False** until §5 gates pass
- GGUF/HF/Ollama only when `EXPORT=True`


## 1. Config

In [ ]:
import os, json, re
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

USE_CPT_MERGE = False
STOCK_MODEL = "unsloth/Qwen3.5-4B-Base"
GATE0_PATH = "/kaggle/input/datasets/rafaelvieira1/theology-cpt-v2/theology_cpt_v2_merged_hf"
BASE_MODEL = GATE0_PATH if USE_CPT_MERGE else STOCK_MODEL

LORA_DIR = Path("/kaggle/working/spurgeon_qa_lora_v2/lora")
TEST_JSONL = Path("/kaggle/input/datasets/spurgeon-qa-mix-v1/qa_test_frozen.jsonl")
if not TEST_JSONL.exists():
    TEST_JSONL = Path("../../data/qa_test_frozen.jsonl")

OUT_METRICS = Path("/kaggle/working/sft_eval_metrics.json")
OUT_MERGED = Path("/kaggle/working/spurgeon_qa_v2_merged_hf")
HF_REPO = "rafaelvieirar1r/qwen3.5-4b-spurgeon-qa"

# Gate: set True only after §5 criteria pass manually
EXPORT = False
MAX_SEQ_LENGTH = 4096

REFUSAL_RE = re.compile(r"does not contain|could not find|insufficient|cannot answer", re.I)
CORRUPT_RE = re.compile(r"pist|spep|RGAR|据", re.I)


## 2. Load model for inference

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
if LORA_DIR.exists():
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, str(LORA_DIR))
FastLanguageModel.for_inference(model)

def generate(messages, max_new_tokens=400):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>"),
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    if "<|im_end|>" in text:
        text = text.split("<|im_end|>")[0]
    return text.strip()


## 3. Frozen battery metrics

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def is_refusal(text):
    return bool(REFUSAL_RE.search(text))

items = load_jsonl(TEST_JSONL)
metrics = {"n": len(items), "format_ok": 0, "echo": 0, "corrupt": 0, "refusal_hits": 0, "refusal_total": 0}
samples = []

for ex in items[: min(50, len(items))]:
    msgs = ex["messages"]
    gold_refusal = is_refusal(msgs[-1]["content"])
    pred = generate(msgs[:-1])
    samples.append({"q": msgs[1]["content"][:200], "pred": pred[:400]})

    if CORRUPT_RE.search(pred):
        metrics["corrupt"] += 1
    if "CONTEXT:" in pred and pred.count("CONTEXT:") > 1:
        metrics["echo"] += 1
    if not CORRUPT_RE.search(pred) and len(pred) > 20:
        metrics["format_ok"] += 1
    if gold_refusal:
        metrics["refusal_total"] += 1
        if is_refusal(pred):
            metrics["refusal_hits"] += 1

metrics["refusal_accuracy"] = (
    metrics["refusal_hits"] / metrics["refusal_total"] if metrics["refusal_total"] else None
)
metrics["corrupt_rate"] = metrics["corrupt"] / max(1, len(samples))
OUT_METRICS.write_text(json.dumps({"metrics": metrics, "samples": samples}, indent=2), encoding="utf-8")
print(json.dumps(metrics, indent=2))
print("Samples written to", OUT_METRICS)


## 4. Gated merge + GGUF export

In [ ]:
if not EXPORT:
    print("EXPORT=False — skip merge/GGUF/HF upload until §5 gates pass.")
else:
    print("Merging 16-bit to", OUT_MERGED)
    model.save_pretrained_merged(str(OUT_MERGED), tokenizer, save_method="merged_16bit")
    tokenizer.save_pretrained(str(OUT_MERGED))
    print(
        "Next: convert to GGUF (f16 + Q4_K_M), upload to", HF_REPO,
        "\nFiles: spurgeon-qa-v2.F16.gguf / spurgeon-qa-v2.Q4_K_M.gguf",
        "\nOllama: fine_tuning/models/Modelfile.qwen35-spurgeon-qa-v2",
        "\nSmoke: python fine_tuning/scripts/smoke_test_ollama.py --model spurgeon-qa-v2",
    )
